In [7]:
import ee
import geemap

ee.Initialize(project='rsga-496805')

point = ee.Geometry.Point([121.5, 24.1])
elev = ee.Image('USGS/SRTMGL1_003') \
    .sample(point, 30) \
    .first() \
    .get('elevation') \
    .getInfo()

print(f"✓ GEE connected — elevation: {elev} m")

✓ GEE connected — elevation: 1496 m


In [8]:
def harmonize_l57(image):
    return image.select(
        ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'],
        ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
    )

def harmonize_l89(image):
    return image.select(
        ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
        ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
    )

In [9]:
def mask_landsat_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0)
    shadow_mask = qa.bitwiseAnd(1 << 4).eq(0)
    return image.updateMask(cloud_mask.And(shadow_mask))

In [10]:
def add_indices(image):
    ndvi = image.normalizedDifference(['NIR', 'Red']).rename('NDVI')
    mndwi = image.normalizedDifference(['Green', 'SWIR1']).rename('MNDWI')
    nbr = image.normalizedDifference(['NIR', 'SWIR2']).rename('NBR')
    return image.addBands([ndvi, mndwi, nbr])